In [1]:
import sys
sys.path.insert(0, "/home/ci2dt2-ai/Proyectos/Psiquiatria")
import kd_common as kdc
import pandas as pd
import numpy as np
import pickle
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import shap

cfg = kdc.get_config("ansiedad")
kdc.set_global_seed(kdc.SEED)

master = pd.read_csv(cfg.experiment_dir / "data" / "master_table.csv", dtype={"patient_id": str, "audio_id": str})
split = pd.read_csv(cfg.experiment_dir / "data" / "development_test_split.csv", dtype={"patient_id": str})
folds = pd.read_csv(cfg.experiment_dir / "data" / "folds_5cv.csv", dtype={"patient_id": str})
dev = kdc.build_dev_frame(master, split, folds)

teacher_test = pd.read_csv(cfg.experiment_dir / "teacher" / "teacher_predictions_test.csv", dtype={"patient_id": str})
preds = {fs: pd.read_csv(cfg.experiment_dir / "features" / fs / "predictions_test.csv", dtype={"patient_id": str}) for fs in ["librosa", "egemaps"]}
print({fs: p.shape for fs, p in preds.items()})

{'librosa': (986, 5), 'egemaps': (986, 5)}


## Fase 16 — Bootstrap agrupado por paciente (1000 iteraciones)

In [2]:
test_df = teacher_test[["sample_id", "patient_id", "y_real", "p_teacher_test"]].copy()
test_df = test_df.rename(columns={"p_teacher_test": "p_teacher"})
for fs in ["librosa", "egemaps"]:
    p = preds[fs][["sample_id", "p_base_test", "p_kd_test"]].rename(
        columns={"p_base_test": f"p_base_{fs}", "p_kd_test": f"p_kd_{fs}"})
    test_df = test_df.merge(p, on="sample_id", how="inner")

model_cols = ["p_teacher"] + [f"p_{m}_{fs}" for fs in ["librosa", "egemaps"] for m in ["base", "kd"]]
summary, boot_scores = kdc.bootstrap_ci(test_df, model_cols, n_boot=1000)
summary.to_csv(cfg.experiment_dir / "results" / "bootstrap_results.csv", index=False)
print(summary.to_string(index=False))

print()
for fs in ["librosa", "egemaps"]:
    delta = boot_scores[f"p_kd_{fs}"] - boot_scores[f"p_base_{fs}"]
    valid = delta[~np.isnan(delta)]
    print(f"{fs}: Delta AUC (KD-Base) = {np.nanmean(delta):.3f} "
          f"[IC95% {np.nanpercentile(delta,2.5):.3f}, {np.nanpercentile(delta,97.5):.3f}] "
          f"-> KD gana en {(valid>0).mean()*100:.0f}% de los remuestreos validos (n={len(valid)}/{len(delta)})")

         model  roc_auc_mean   ci_low  ci_high  n_valid_resamples
     p_teacher      0.297658 0.030952 0.655410                994
p_base_librosa      0.523146 0.446625 0.576258                994
  p_kd_librosa      0.500000 0.500000 0.500000                994
p_base_egemaps      0.649507 0.542503 0.750469                994
  p_kd_egemaps      0.467562 0.203915 0.701020                994

librosa: Delta AUC (KD-Base) = -0.023 [IC95% -0.076, 0.053] -> KD gana en 22% de los remuestreos validos (n=994/1000)
egemaps: Delta AUC (KD-Base) = -0.182 [IC95% -0.410, 0.033] -> KD gana en 6% de los remuestreos validos (n=994/1000)


## Fase 17 — SHAP: Base vs KD por feature_set

Se usa `shap.Explainer` genérico (funciona con cualquier modelo del barrido lazypredict,
no solo árboles) sobre una muestra de Desarrollo, con un fondo (background) reducido para
que sea rápido independientemente de qué familia de modelo haya ganado.

In [3]:
rng = np.random.RandomState(kdc.SEED)

for fs in ["librosa", "egemaps"]:
    out_dir = cfg.experiment_dir / "explainability"
    fdir = cfg.experiment_dir / "features" / fs
    with open(fdir / "dev_feature_medians.pkl", "rb") as f:
        medians = pickle.load(f)
    X_dev, feat_cols, _ = kdc.load_feature_matrix(cfg, fs, dev, medians=medians)

    sample_idx = rng.choice(len(X_dev), size=min(120, len(X_dev)), replace=False)
    bg_idx = sample_idx[:40]
    explain_idx = sample_idx[40:120]
    X_bg = X_dev.iloc[bg_idx]
    X_explain = X_dev.iloc[explain_idx]

    for kind, fname in [("base", "base_final_model.pkl"), ("kd", "kd_final_model.pkl")]:
        with open(fdir / fname, "rb") as f:
            model = pickle.load(f)
        predict_fn = model.predict_proba if hasattr(model, "predict_proba") else model.predict

        def f_wrap(X, _predict_fn=predict_fn):
            X = pd.DataFrame(X, columns=feat_cols)
            out = _predict_fn(X)
            return out[:, 1] if out.ndim == 2 else out

        explainer = shap.Explainer(f_wrap, X_bg, algorithm="permutation", max_evals=2 * len(feat_cols) + 1)
        sv = explainer(X_explain)

        mean_abs = pd.DataFrame({"feature": feat_cols, "mean_abs_shap": np.abs(sv.values).mean(axis=0)})
        mean_abs = mean_abs.sort_values("mean_abs_shap", ascending=False)
        mean_abs.to_csv(out_dir / f"shap_{fs}_{kind}.csv", index=False)

        top = mean_abs.head(15).iloc[::-1]
        plt.figure(figsize=(6, 5))
        plt.barh(top["feature"], top["mean_abs_shap"])
        plt.xlabel("mean(|SHAP value|)")
        plt.title(f"{fs} - {kind}")
        plt.tight_layout()
        plt.savefig(out_dir / f"shap_{fs}_{kind}.png", dpi=110)
        plt.close()
        print(f"{fs}/{kind}: top feature = {mean_abs.iloc[0]['feature']}")

print("SHAP guardado en explainability/")

librosa/base: top feature = lpcc8_mean


librosa/kd: top feature = mfcc1_mean


egemaps/base: top feature = slopeV0-500_sma3nz_amean


egemaps/kd: top feature = F0semitoneFrom27.5Hz_sma3nz_percentile20.0
SHAP guardado en explainability/
